
# ERE Consolidated Pipeline (ready‑to‑run)

**Purpose.** One notebook for the *circularity paradox* workflow:

1. Ingest & clean raw marketplace data (SMII) → `processed/df_cleaned_en.parquet`  
2. Category & market summaries (SMIII) → `derived/*`  
3. Representative products, prices, and economic benefits (PC merged) → `derived/*`  
4. Household CF factors for Sweden → `factors/*`  
5. Potential environmental benefits (IO‑LCA) → `derived/SI_V_pot_env_ben.parquet`  
6. Re‑spending effects → `derived/df_re_spending_effect_{agg,disagg}.parquet`  
7. ERE computation → `results/df_ere.parquet` (+ optional exports)

**Folders (auto‑created).** `raw/`, `processed/`, `derived/`, `factors/`, `results/`, `exports/`, `diagnostics/`

> Internal data uses **Parquet**. CSV/Excel are written only to `exports/` on demand.


## 00. Config & utilities

In [1]:

# 00. Config & utilities
from pathlib import Path
import pandas as pd
import numpy as np

RUN_ID = "2025-10-29"

BASE = Path(".")
RAW = BASE/"raw"
PROC = BASE/"processed"
DER = BASE/"derived"
FAC = BASE/"factors"
RES = BASE/"results"
EXP = BASE/"exports"
DIA = BASE/"diagnostics"

for p in [RAW, PROC, DER, FAC, RES, EXP, DIA]:
    p.mkdir(parents=True, exist_ok=True)

def write_df(df: pd.DataFrame, path: Path, enforce=False):
    if path.exists() and not enforce:
        raise FileExistsError(f"{path} exists (set enforce=True to overwrite).")
    if path.suffix == ".parquet":
        df.to_parquet(path, index=False)
    elif path.suffix == ".csv":
        df.to_csv(path, index=False)
    elif path.suffix in (".feather", ".arrow"):
        df.to_feather(path)
    else:
        raise ValueError("Use .parquet or .csv/.feather")


## 01 Ingest & Clean (SMII)

In [2]:
# TODO: implement clean(...)
# df_raw = pd.read_excel(RAW/'scraped_data.xlsx')
# df_clean = clean(df_raw)
# write_df(df_clean, PROC/'df_cleaned_en.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [3]:
import requests
import bs4
from bs4 import BeautifulSoup
import xlsxwriter
import re
import pandas as pd
import numpy as np
import six

In [4]:
#website has changed its html structure, therefore this script does not longer work 

workbook = xlsxwriter.Workbook('dataset/data_privat_collected 2019-10-28.xlsx')
worksheet = workbook.add_worksheet()
bold = workbook.add_format({'bold': 1})
worksheet.set_column(0, 0, 50)
worksheet.set_column(1, 1, 30)
worksheet.set_column(2, 2, 20)
worksheet.set_column(3, 3, 20)
worksheet.write_string(0, 0, "Item", bold)
worksheet.write_string(0, 1, "Price", bold)
worksheet.write_string(0, 2, "Category", bold)
worksheet.write_string(0, 3, "Region", bold)

row = 1
col = 0

url = 'https://www.blocket.se/hela_sverige?ca=11&st=s&f=p&w=3'

while not url.endswith('&last=1'):  
    
    response = requests.get(url)    
    try:    
        response.raise_for_status() 
    except Exception as exc:    
        print('There was a problem: %s' % (exc))

    soup = BeautifulSoup(response.text, "html.parser")

    items = soup.find_all("a", "item_link")
    prices = soup.find_all("p", "list_price") 
    categories = soup.find_all('a', {'tabindex': '-1'})
    regions = soup.find_all('div', 'pull-left')

    for item_a, price_p, category_a, region_div in zip(items, prices, categories, regions[6:]):
         item = item_a.string.strip()   

         price = price_p.text
         if not price:
             price = "NULL"

         category = category_a.text
         if not(re.search(r"Lägenheter|Utland|Djur|Villor|Tjänster|Lokaler & fastigheter|Affärsöverlåtelser|Upplevelser & nöje|Tomter|Gårdar|Efterlysningar|Fritidsboende|Radhus", category)) and not(re.search(r"Jobb", region_div.text)):
             region = region_div.text.split(',')[-1]          
             worksheet.write_string(row, col, item)
             worksheet.write_string(row, col + 1, price)
             worksheet.write_string(row, col + 2, category)        
             worksheet.write_string(row, col + 3, region)             
             row += 1     
         
    nextLink = soup.find_all('a', 'page_nav')[5]
    if not "Nästa sida »" in nextLink.decode_contents().strip():
        nextLink = soup.find_all('a', 'page_nav')[6]
        if not "Nästa sida »" in nextLink.decode_contents().strip():  
            nextLink = soup.find_all('a', 'page_nav')[7]               
    
    nextLinkSuffix = nextLink.get('href')     
    url = 'https://www.blocket.se/hela_sverige' + nextLinkSuffix     

workbook.close() 

IndexError: list index out of range

In [ ]:
#Opening scraped data
# This is the supporting material I
df = pd.read_excel(r'processed/scraped_data.xlsx') 

In [ ]:
df.count()

In [ ]:
#Converting Price from string to float:
df_dirty = df.copy()
df_dirty['Price'] = df_dirty['Price'].str.replace('kr', '').str.replace(' ', '').astype(float)

In [ ]:
#Removing rows with NaN values from the Price column:
df_dirty = df_dirty[pd.notnull(df_dirty['Price'])]

In [ ]:
#Glancing at the maximum values for the Prices by Category it is possible to notice that outliers ought to be removed:
df_dirty.groupby('Category')['Price'].describe()

In [ ]:
#Removing outliers, i.e. prices that differ significantly from other observations:
df_cleaned = pd.DataFrame(columns=df.columns)

for category in df_dirty['Category'].unique().tolist(): 
    df_category = df_dirty.loc[df_dirty['Category'] == category].copy()
    
    #removing the bottom and top 3% with a quantile filter of each category
    #https://stackoverflow.com/questions/23199796/detect-and-exclude-outliers-in-a-pandas-dataframe/43093390#43093390
    q_low = df_category['Price'].quantile(0.03) 
    q_hi = df_category['Price'].quantile(0.97)
    df_filtered = df_category[(df_category["Price"] < q_hi) & (df_category["Price"] > q_low)] 

    #df_category = df_category[np.abs(df_category['Price']-df_category['Price'].mean()) <= (3*df_category['Price'].std())]
    df_cleaned = df_cleaned.append(df_filtered)


df_cleaned.sort_index()

In [ ]:
#visualizing statictics about the new dataframe
df_cleaned.groupby('Category')['Price'].describe()

In [ ]:
#Number of rows removed due to outliers:
df_dirty.count()-df_cleaned.count()

In [ ]:
#Variable df receives the cleaned data:
df = df_cleaned.copy()

In [ ]:
# number of Regions
df['Region'].value_counts().count()

In [ ]:
df['Region'].value_counts()


In [ ]:
#Number of categories
df['Category'].value_counts().count()


In [ ]:
dic_cat = {'Motorcyklar': 'Motorcycles',
'Bildelar & biltillbehör': 'Car parts & car accessories',
'Möbler & heminredning': 'Furniture & home decor',
'Sport- & fritidsutrustning': 'Sports & leisure equipment',
'Barnkläder & skor': 'Children\'s clothing & shoes',
'MC-delar & tillbehör': 'Motorcycle parts & accessories',
'Ljud & bild': 'Audio & video',
'Bilar': 'Cars',
'Datorer & TV-spel': 'Computers & video games',
'Bygg & trädgård': 'Construction & garden',
'Övrigt': 'Other',
'Husgeråd & vitvaror': 'Housewares & appliances',
'Kläder & skor': 'Clothes & shoes',
'Barnartiklar & leksaker': 'Children\'s articles & toys',
'Husvagnar & husbilar': 'Caravans & campers',
'Skogs- & lantbruksmaskiner': 'Forestry & agricultural machinery',
'Musikutrustning': 'Music equipment',
'Inventarier & maskiner': 'Fixtures & machines',
'Telefoner & tillbehör': 'Phones & accessories',
'Böcker & studentlitteratur': 'Books & student literature',
'Verktyg': 'Tools',
'Hästar & ridsport': 'Horses & equestrian',
'Båtdelar & tillbehör': 'Boat parts & accessories',
'Mopeder & A-traktor': 'Mopeds & doodlebug tractors',
'Cyklar': 'Bicycles',
'Lastbil, truck & entreprenad': 'Trucks & construction',
'Båtar': 'Boats',
'Snöskotrar': 'Snowmobiles',
'Accessoarer & klockor': 'Accessories & watches',
'Hobby & samlarprylar': 'Hobby & collectibles',
'Jakt & fiske': 'Hunting & fishing',
'Snöskoterdelar & tillbehör': 'Snowmobile parts & accessories'}

In [ ]:
#chainging names of categories to English
df_en=df.replace({"Category": dic_cat})
df_en

In [ ]:
df_en.to_csv('processed/df_cleaned_en.parquet', index=False)

In [ ]:
#Opening scraped data
# This is the supporting material I
df_en = pd.read_csv(r'processed/df_cleaned_en.parquet') 

In [ ]:
#visualizing statictics about the new dataframe
df_en_described = df_en.groupby('Category')['Price'].describe()
df_en_described

In [ ]:
df_en_described.to_csv('processed/df_cleaned_en_described.csv')

## 02 Category & Market Summaries (SMIII)

In [ ]:
# TODO: implement build_category_overview(...), build_regional_markets(...)
# df = pd.read_parquet(PROC/'df_cleaned_en.parquet')
# cats = build_category_overview(df); write_df(cats, DER/'product_categories_overview.parquet')
# markets = build_regional_markets(df); write_df(markets, DER/'regional_markets.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
import pandas as pd

In [ ]:
# Filter and display only the duplicated rows
duplicated_rows = df_en[df_en.duplicated()]

print("Duplicated Rows:")
print(duplicated_rows)

In [ ]:
#import data collected and cleaned from supporting material II: data collection and cleaning
df_en = pd.read_csv('processed/df_cleaned_en.parquet')
df_en

In [ ]:
duplicates_count = df_en.duplicated().sum()

print(f"Number of duplicated rows: {duplicates_count}")

In [ ]:
#computing statistics
df_pc_stats = df_en.groupby(['Category'])['Price'].describe()
df_p_sum=df_en.groupby('Category')['Price'].sum()


In [ ]:
pco = pd.concat([df_pc_stats, df_p_sum], axis=1) # joining dataframes
pco = pco.rename(columns = {'count':'# of products', 'mean':'Average product price [SEK]', 'Price' : 'Total value [SEK]'}) #renaming columns

In [ ]:
# rank for # of products
pco=pco.sort_values(by=['# of products'], ascending=False)

pco['Rank by # of products']=pco.reset_index().index + 1

pco['% of the total # of products'] = (pco['# of products']/pco['# of products'].sum()).round(4)*100


In [ ]:
# rank for average price
pco=pco.sort_values(by=['Average product price [SEK]'], ascending=False)

pco['Rank by average product price']=pco.reset_index().index + 1


In [ ]:
#rank for total value
pco=pco.sort_values(by=['Total value [SEK]'], ascending=False)

pco['Rank by total value [SEK]']=pco.reset_index().index + 1

pco['% of total value'] = (pco['Total value [SEK]']/pco['Total value [SEK]'].sum()).round(4)*100

In [ ]:
#sort the data by number of products
pco=pco.sort_values(by=['Rank by # of products'])

In [ ]:
#reorder columns
pco = pco[['# of products', 'Rank by # of products', '% of the total # of products', 'Total value [SEK]', 
    'Rank by total value [SEK]', '% of total value', 'Average product price [SEK]', 'Rank by average product price',
    'std',	'min',	'25%',	'50%',	'75%',	'max' ]]

pco

In [ ]:
#Save dataframe
#merge with results/_pot_env_ben.xlsx obtained in SI_VI_pot_env_ben.ipynb, and include as a table in results/.
pco.to_excel('processed/product_categories_overview.xlsx')

In [ ]:
#Number of regions
df_en['Region'].value_counts().count()

In [ ]:
# computing the statistics for the overview dataframe

df_count=df_en.groupby('Region')['Price'].count()
df_mean=df_en.groupby('Region')['Price'].mean().round(2)
#df_std=df_en.groupby('County')['Price'].std().round(2)
df_sum=df_en.groupby('Region')['Price'].sum().round(2)


#pco=pd.concat([df_count, df_mean, df_std, df_sum], axis=1)
rmo=pd.concat([df_count, df_mean, df_sum], axis=1)
#pco.columns=['# of products', 'Average product price', 'Standard deviation', 'Total value']
rmo.columns=['# of products', 'Average product price [SEK]', 'Total value [SEK]']


In [ ]:
# rank for number of products

rmo=rmo.sort_values(by=['# of products'], ascending=False)

rmo['Rank by # of products']=rmo.reset_index().index + 1

rmo['% of the total # of products'] = (rmo['# of products']/rmo['# of products'].sum()).round(4)*100


In [ ]:
# rank for average price
rmo=rmo.sort_values(by=['Average product price [SEK]'], ascending=False)

rmo['Rank by average product price']=rmo.reset_index().index + 1

In [ ]:
# rank for total value

rmo=rmo.sort_values(by=['Total value [SEK]'], ascending=False)

rmo['Rank by total value']=rmo.reset_index().index + 1

rmo['% of total value'] = (rmo['Total value [SEK]']/rmo['Total value [SEK]'].sum()).round(4)*100


In [ ]:
rmo = rmo[['# of products', 'Rank by # of products',	'% of the total # of products', 
    'Total value [SEK]', 'Rank by total value',	'% of total value',
    'Average product price [SEK]', 'Rank by average product price' 
    ]]

In [ ]:
# sort by number of products

rmo=rmo.sort_values(by=['Rank by # of products'])

rmo

In [ ]:

rmo.to_excel('processed/regional_markets_overview.xlsx')

In [ ]:
rm_count=df_en.groupby(['Region','Category']).size().reset_index(name='counts')
rm_count

In [ ]:
rm_count.to_excel('processed/regional_markets_counts.xlsx', index=False)

## 03 Representative Products & Prices (PC merged) – prefer SI_VII_PC_econ_ben; fold parts of 20240116

In [ ]:
# TODO: implement make_representatives_and_prices(...)
# df = pd.read_parquet(PROC/'df_cleaned_en.parquet')
# cats = pd.read_parquet(DER/'product_categories_overview.parquet')
# rp, prices, totals, econ_ben = make_representatives_and_prices(df, cats)
# write_df(rp, DER/'representative_products.parquet')
# write_df(prices, DER/'resale_retail_product_prices.parquet')
# write_df(totals, DER/'total_values_per_category.parquet')
# write_df(econ_ben, DER/'PC_ecom_ben.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
#load packages
import pandas as pd
import numpy as np

In [ ]:
#import data collected and cleaned from supporting material II
df_en = pd.read_csv(r'processed/df_cleaned_en.parquet')
df_en

In [ ]:
# Selecting the products with the closest price to the price average of each category

df_RP = pd.DataFrame(columns=df_en.columns)

for category in df_en['Category'].unique().tolist():   
    
    df_category = df_en.loc[df_en['Category'] == category].copy() 
    mean_category = df_category['Price'].mean()
    df_RP = df_RP.append(df_category.iloc[(df_category['Price']-mean_category).abs().argsort()[:1]])

df_RP = df_RP.sort_values(by ='Category')    

In [ ]:
df_RP

In [ ]:

df_RP = df_RP.rename(columns = {'Price':'Price used product [SEK]'}) #renaming column price
df_RP.style.hide_index()

In [ ]:
# Price new product and source
df_RP.loc[df_RP['Item']=='Raymond Weil klocka', 'Source (price new product)'] =  'klockorochsmycken.se'
df_RP.loc[df_RP['Item']=='Raymond Weil klocka', 'Price new product [SEK]'] = 9850

df_RP.loc[df_RP['Item']=='Nikon d5200 paket', 'Source (price new product)'] =  'ernstsonfoto.se'
df_RP.loc[df_RP['Item']=='Nikon d5200 paket', 'Price new product [SEK]'] = 7895

df_RP.loc[df_RP['Item']=='MTB Cresent Draupner m trainer', 'Source (price new product)'] =  'cykelaffaren.se'
df_RP.loc[df_RP['Item']=='MTB Cresent Draupner m trainer', 'Price new product [SEK]'] = 6990

df_RP.loc[df_RP['Item']=='Segelbåtsvagn', 'Source (price new product)'] =  'gardsman.se'
df_RP.loc[df_RP['Item']=='Segelbåtsvagn', 'Price new product [SEK]'] = 18495

df_RP.loc[df_RP['Item']=='H-35 segelbåt', 'Source (price new product)'] =  'sailguide.com'
df_RP.loc[df_RP['Item']=='H-35 segelbåt', 'Price new product [SEK]'] = 220000

df_RP.loc[df_RP['Item']=='Verkstadshandböcker till Dodge 1958 - 1959', 'Source (price new product)'] =  'hortlund.se'
df_RP.loc[df_RP['Item']=='Verkstadshandböcker till Dodge 1958 - 1959', 'Price new product [SEK]'] = 995

df_RP.loc[df_RP['Item']=='S Däck', 'Source (price new product)'] =  'gummihuset.se'
df_RP.loc[df_RP['Item']=='S Däck', 'Price new product [SEK]'] = 12788

df_RP.loc[df_RP['Item']=='Hobby KMFE 560 2008 De Luxe Barnkammarvagn', 'Source (price new product)'] =  'holmgrensbil.se'
df_RP.loc[df_RP['Item']=='Hobby KMFE 560 2008 De Luxe Barnkammarvagn', 'Price new product [SEK]'] = 269900

df_RP.loc[df_RP['Item']=='Audi A6 2,0 TDI -11', 'Source (price new product)'] =  'audistockholm.se'
df_RP.loc[df_RP['Item']=='Audi A6 2,0 TDI -11', 'Price new product [SEK]'] = 359000

df_RP.loc[df_RP['Item']=='Ergobaby 360 omni', 'Source (price new product)'] =  'lekmer.se'
df_RP.loc[df_RP['Item']=='Ergobaby 360 omni', 'Price new product [SEK]'] = 1424

df_RP.loc[df_RP['Item']=='Tommy Hilfiger kavaj', 'Source (price new product)'] =  'zalando.se'
df_RP.loc[df_RP['Item']=='Tommy Hilfiger kavaj', 'Price new product [SEK]'] = 458

df_RP.loc[df_RP['Item']=='Signerad matchtröja', 'Source (price new product)'] =  'nike.se'
df_RP.loc[df_RP['Item']=='Signerad matchtröja', 'Price new product [SEK]'] = 1599

df_RP.loc[df_RP['Item']=='Acer Chromebook CB5-312T', 'Source (price new product)'] =  'netonnet.se'
df_RP.loc[df_RP['Item']=='Acer Chromebook CB5-312T', 'Price new product [SEK]'] = 6490

df_RP.loc[df_RP['Item']=='Fönster nya 4 st. 100 cm x 1625 cm', 'Source (price new product)'] =  'hemfint.se'
df_RP.loc[df_RP['Item']=='Fönster nya 4 st. 100 cm x 1625 cm', 'Price new product [SEK]'] = 4395

df_RP.loc[df_RP['Item']=='Kemppi minarc 220', 'Source (price new product)'] =  'svetsochtillbehor.se'
df_RP.loc[df_RP['Item']=='Kemppi minarc 220', 'Price new product [SEK]'] = 16900

df_RP.loc[df_RP['Item']=='Ferguson 65 med lastare och snökedjor', 'Source (price new product)'] =  'mascus.se'
df_RP.loc[df_RP['Item']=='Ferguson 65 med lastare och snökedjor', 'Price new product [SEK]'] = 39818

df_RP.loc[df_RP['Item']=='Soffbord glas och mässing', 'Source (price new product)'] =  'royaldesign.se'
df_RP.loc[df_RP['Item']=='Soffbord glas och mässing', 'Price new product [SEK]'] = 5610

df_RP.loc[df_RP['Item']=='Märklin', 'Source (price new product)'] =  'cdon.se'
df_RP.loc[df_RP['Item']=='Märklin', 'Price new product [SEK]'] = 3899

df_RP.loc[df_RP['Item']=='Uthyres du som har stort häst intresse', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Uthyres du som har stort häst intresse', 'Price new product [SEK]'] = 1.5*6300

df_RP.loc[df_RP['Item']=='Kondens Torktumlare Av AEG LAVATHERM', 'Source (price new product)'] =  'computersalg.se'
df_RP.loc[df_RP['Item']=='Kondens Torktumlare Av AEG LAVATHERM', 'Price new product [SEK]'] = 8234

df_RP.loc[df_RP['Item']=='Älgstudsare', 'Source (price new product)'] =  'torsbohandels.se'
df_RP.loc[df_RP['Item']=='Älgstudsare', 'Price new product [SEK]'] = 7999

df_RP.loc[df_RP['Item']=='Yamaha Aerox Naked R 2014', 'Source (price new product)'] =  'ahlqvistmc.se'
df_RP.loc[df_RP['Item']=='Yamaha Aerox Naked R 2014', 'Price new product [SEK]'] = 27490

df_RP.loc[df_RP['Item']=='Helt nya Rukka Gore-Tex Byxor', 'Source (price new product)'] =  'tradeinn.se'
df_RP.loc[df_RP['Item']=='Helt nya Rukka Gore-Tex Byxor', 'Price new product [SEK]'] = 8902

df_RP.loc[df_RP['Item']=='SMC Wildcat 500 -15', 'Source (price new product)'] =  'pvlv.se'
df_RP.loc[df_RP['Item']=='SMC Wildcat 500 -15', 'Price new product [SEK]'] = 64900

df_RP.loc[df_RP['Item']=='Marshall DSL40C Vintage', 'Source (price new product)'] =  'soundstorexl.se'
df_RP.loc[df_RP['Item']=='Marshall DSL40C Vintage', 'Price new product [SEK]'] = 7899

df_RP.loc[df_RP['Item']=='Villa uthyres på Kärna, 3750 sek/mån', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Villa uthyres på Kärna, 3750 sek/mån', 'Price new product [SEK]'] = 1.5*3750

df_RP.loc[df_RP['Item']=='Apple Watch 3 42mm Cellular Milanese', 'Source (price new product)'] =  'computersalg.se'
df_RP.loc[df_RP['Item']=='Apple Watch 3 42mm Cellular Milanese', 'Price new product [SEK]'] = 6765

df_RP.loc[df_RP['Item']=='SLP pipa och slutburk', 'Source (price new product)'] =  'skoterdelen.se'
df_RP.loc[df_RP['Item']=='SLP pipa och slutburk', 'Price new product [SEK]'] = 8995

df_RP.loc[df_RP['Item']=='Lynx Adventure LX 600 ACE-11', 'Source (price new product)'] =  'brplynx.com'
df_RP.loc[df_RP['Item']=='Lynx Adventure LX 600 ACE-11', 'Price new product [SEK]'] = 134900

df_RP.loc[df_RP['Item']=='Edea Overture strl 210, vanligt storlek 31,5', 'Source (price new product)'] =  'xxl.se'
df_RP.loc[df_RP['Item']=='Edea Overture strl 210, vanligt storlek 31,5', 'Price new product [SEK]'] = 4689

df_RP.loc[df_RP['Item']=='Kit för Ac rep och fyllning', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Kit för Ac rep och fyllning', 'Price new product [SEK]'] = 1.5*2900

df_RP.loc[df_RP['Item']=='El truck -01', 'Source (price new product)'] =  'maskinexperten.se'
df_RP.loc[df_RP['Item']=='El truck -01', 'Price new product [SEK]'] = 294900

In [ ]:
df_RP['Value retained (%)'] = (df_RP['Price used product [SEK]'] / df_RP['Price new product [SEK]'])*100

In [ ]:
#reorder columns

df_RP = df_RP[['Item',	'Price used product [SEK]',	'Price new product [SEK]', 'Value retained (%)', 'Source (price new product)', 'Category',	'Region']]
df_RP.style.hide_index()

In [ ]:
#save df_RP to csv
df_RP.to_csv('processed/representative_products.csv', index=False)

In [ ]:
#save df_RP to excell to include in the supplementary information IV
df_RP.to_excel('processed/representative_products_to_SI_IV.xlsx', index=False)

In [ ]:
df_en_described = pd.read_csv(r'processed/df_cleaned_en_described.csv')
df_en_described

In [ ]:
df_en_RP_described_merged = pd.merge(df_RP, df_en_described, how='inner', on = 'Category')
df_en_RP_described_merged

In [ ]:
#keeping only the columns of interest
df_en_RP_method = df_en_RP_described_merged[['Item', 
    'Price used product [SEK]',	
    'Price new product [SEK]',	
    'Value retained (%)',	
    'Source (price new product)',	
    'Category']] 

#renaming the column
df_en_RP_method = df_en_RP_method.rename({
    'Item': 'Representative product',
    'Price used product [SEK]': 'Resale price [SEK]',
    'Price new product [SEK]': 'Retail price [SEK]', 
    'Source (price new product)': 'Source (retail price)'}, 
    axis=1)

#reorder columns
df_en_RP_method = df_en_RP_method[['Category',
    'Representative product',
    'Resale price [SEK]',
    'Retail price [SEK]',
    'Value retained (%)',
    'Source (retail price)']]
df_en_RP_method

In [ ]:
#save for including in the method section of the article
df_en_RP_method.to_excel('processed/df_en_RP_method.xlsx')

In [ ]:
#check point
# load dataframes if needed
df_en = pd.read_csv('processed/df_cleaned_en.parquet') #scraped data
df_RP = pd.read_csv('processed/representative_products.csv') #representative products

In [ ]:
#renaming column price
df_en = df_en.rename(columns = {'Price':'Resale price [SEK]'}) 

In [ ]:
# drop columns not needed
df_RP = df_RP.drop(['Item',	'Price used product [SEK]',	'Price new product [SEK]', 'Source (price new product)', 'Region'], axis=1)

In [ ]:
df_UP = df_en.merge(df_RP, on='Category', how='left')
df_UP


In [ ]:
#The retail prices are computed by dividing the prices of the secondhand items by the value retained (%) of the reference products of each category.
#df['Values'] = df.Price * df.Value_retained
df_UP['Retail price [SEK] (projected)'] = (df_UP['Resale price [SEK]'] / df_UP['Value retained (%)'])*100
df_UP

In [ ]:
#save df_UP to csv
df_UP.to_csv('processed/resale_retail_product_prices.csv', index=False) #this df will be used in the calculation of the potencial environmental benetis

In [ ]:
#load df_UP if necessary
df_UP_disagg = pd.read_csv('processed/resale_retail_product_prices.csv')
df_UP_disagg

In [ ]:
#compute economic savings of buyers
df_UP_disagg['Savings of buyers [SEK]'] = df_UP_disagg['Retail price [SEK] (projected)'] - df_UP_disagg['Resale price [SEK]']

#drop columns that will not be used
columns_to_drop = ['Region','Value retained (%)','Retail price [SEK] (projected)']
df_UP_disagg = df_UP_disagg.drop(columns=columns_to_drop)

#renaming columns
df_UP_disagg = df_UP_disagg.rename(columns={'Resale price [SEK]': 'Earnings of sellers [SEK]'})

df_UP_disagg

In [ ]:
#save to csv
df_UP_disagg.to_csv('processed/diagg_ecom_ben.csv', index=False)

In [ ]:
#load df_UP if necessary
df_UP = pd.read_csv('processed/resale_retail_product_prices.csv')
df_UP

In [ ]:
df_up_cat_grouped = df_UP.groupby(['Category'])
df_up_cat_sum = df_up_cat_grouped[['Resale price [SEK]', 'Retail price [SEK] (projected)']].agg('sum')
df_up_cat_sum.reset_index(inplace=True)

#rename
df_up_cat_sum = df_up_cat_sum.rename(columns = {'Resale price [SEK]':'Total value based on resale price [SEK]'}) 
df_up_cat_sum = df_up_cat_sum.rename(columns = {'Retail price [SEK] (projected)':'Total value based on retail price [SEK] (projected)'}) 

#convert SEK to M SEK
df_up_cat_sum['Total value based on resale price [SEK]'] = df_up_cat_sum['Total value based on resale price [SEK]']/(1000000)
df_up_cat_sum['Total value based on retail price [SEK] (projected)'] = df_up_cat_sum['Total value based on retail price [SEK] (projected)']/(1000000)

#rename
df_up_cat_sum = df_up_cat_sum.rename(columns = {'Total value based on resale price [SEK]':'Total value based on resale price [M SEK]'}) 
df_up_cat_sum = df_up_cat_sum.rename(columns = {'Total value based on retail price [SEK] (projected)':'Total value based on retail price [M SEK] (projected)'}) 

#calculate value retained
df_up_cat_sum['Value retained (%)'] = (df_up_cat_sum['Total value based on resale price [M SEK]'] / df_up_cat_sum['Total value based on retail price [M SEK] (projected)'])*100

df_up_cat_sum

In [ ]:
#save df
df_up_cat_sum.reset_index(drop=True, inplace=True)
df_up_cat_sum.to_csv('processed/total_values_per_category.csv', index=False)

In [ ]:
#load df if nedeed
df_up_cat_sum = pd.read_csv('processed/total_values_per_category.csv')

In [ ]:
# load pco dataset
pco = pd.read_excel('processed/product_categories_overview.xlsx')
# drop columns not needed
pco = pco.drop(['std',	'min',	'25%',	'50%',	'75%',	'max', 'Average product price [SEK]',	'Rank by average product price'], axis=1)
pco

In [ ]:
# merge datasets
df_PC_results/ = pco.merge(df_up_cat_sum, on='Category', how='left')

#delete repeted column and not of interest
df_PC_results/ = df_PC_results/.drop(['Total value [SEK]','Value retained (%)'], axis=1)

#move new column to correct position
column_to_move = 'Total value based on resale price [M SEK]'
desired_position = 4 # 0-indexed
columns = df_PC_results/.columns.tolist() ## Get the current column order
columns.remove(column_to_move) # Remove the column to move from the list of columns
columns.insert(desired_position, column_to_move) # Insert the column at the desired position
df_PC_results/ = df_PC_results/[columns] # Reorder the DataFrame columns

#rename columns
column_rename_dict = {'Total value based on resale price [M SEK]': 'Total value [M SEK] (resale)',
    'Rank by total value [SEK]': 'Rank by total value (resale)',
    '% of total value': '% of total value (resale)',
    'Total value based on retail price [M SEK] (projected)': 'Total value [M SEK] (retail)'
    } # Define a dictionary to map old column names to new names
df_PC_results/.rename(columns=column_rename_dict, inplace=True) # Use the rename() method to rename the columns


df_PC_results/

In [ ]:
#compute total economic savings of buyers
df_PC_results/['Total economic savings of buyers [M SEK]'] = df_PC_results/['Total value [M SEK] (retail)'] - df_PC_results/['Total value [M SEK] (resale)']

#drop column Total value [M SEK] (retail)
df_PC_results/.drop(columns=['Total value [M SEK] (retail)'], inplace=True)

df_PC_results/

In [ ]:
#include rank by total savings
df_PC_results/['Rank by total economic savings'] = df_PC_results/['Total economic savings of buyers [M SEK]'].rank(ascending=False)

## Convert the rank values to integers
df_PC_results/['Rank by total economic savings'] = df_PC_results/['Rank by total economic savings'].astype(int)

df_PC_results/

In [ ]:
#save to csv
df_PC_results/.to_csv('processed/PC_ecom_ben.parquet', index=False)

In [ ]:
#load packages
import pandas as pd
import numpy as np

In [ ]:
#import data collected and cleaned from supporting material II
df_en = pd.read_csv(r'processed/df_cleaned_en.parquet')
df_en

In [ ]:
# Selecting the products with the closest price to the price average of each category

df_RP = pd.DataFrame(columns=df_en.columns)

for category in df_en['Category'].unique().tolist():   
    
    df_category = df_en.loc[df_en['Category'] == category].copy() 
    mean_category = df_category['Price'].mean()
    df_RP = df_RP.append(df_category.iloc[(df_category['Price']-mean_category).abs().argsort()[:1]])

df_RP = df_RP.sort_values(by ='Category')    

In [ ]:
df_RP

In [ ]:

df_RP = df_RP.rename(columns = {'Price':'Price used product [SEK]'}) #renaming column price
df_RP.style.hide_index()

In [ ]:
# Price new product and source
df_RP.loc[df_RP['Item']=='Raymond Weil klocka', 'Source (price new product)'] =  'klockorochsmycken.se'
df_RP.loc[df_RP['Item']=='Raymond Weil klocka', 'Price new product [SEK]'] = 9850

df_RP.loc[df_RP['Item']=='Nikon d5200 paket', 'Source (price new product)'] =  'ernstsonfoto.se'
df_RP.loc[df_RP['Item']=='Nikon d5200 paket', 'Price new product [SEK]'] = 7895

df_RP.loc[df_RP['Item']=='MTB Cresent Draupner m trainer', 'Source (price new product)'] =  'cykelaffaren.se'
df_RP.loc[df_RP['Item']=='MTB Cresent Draupner m trainer', 'Price new product [SEK]'] = 6990

df_RP.loc[df_RP['Item']=='Segelbåtsvagn', 'Source (price new product)'] =  'gardsman.se'
df_RP.loc[df_RP['Item']=='Segelbåtsvagn', 'Price new product [SEK]'] = 18495

df_RP.loc[df_RP['Item']=='H-35 segelbåt', 'Source (price new product)'] =  'sailguide.com'
df_RP.loc[df_RP['Item']=='H-35 segelbåt', 'Price new product [SEK]'] = 220000

df_RP.loc[df_RP['Item']=='Verkstadshandböcker till Dodge 1958 - 1959', 'Source (price new product)'] =  'hortlund.se'
df_RP.loc[df_RP['Item']=='Verkstadshandböcker till Dodge 1958 - 1959', 'Price new product [SEK]'] = 995

df_RP.loc[df_RP['Item']=='S Däck', 'Source (price new product)'] =  'gummihuset.se'
df_RP.loc[df_RP['Item']=='S Däck', 'Price new product [SEK]'] = 12788

df_RP.loc[df_RP['Item']=='Hobby KMFE 560 2008 De Luxe Barnkammarvagn', 'Source (price new product)'] =  'holmgrensbil.se'
df_RP.loc[df_RP['Item']=='Hobby KMFE 560 2008 De Luxe Barnkammarvagn', 'Price new product [SEK]'] = 269900

df_RP.loc[df_RP['Item']=='Audi A6 2,0 TDI -11', 'Source (price new product)'] =  'audistockholm.se'
df_RP.loc[df_RP['Item']=='Audi A6 2,0 TDI -11', 'Price new product [SEK]'] = 359000

df_RP.loc[df_RP['Item']=='Ergobaby 360 omni', 'Source (price new product)'] =  'lekmer.se'
df_RP.loc[df_RP['Item']=='Ergobaby 360 omni', 'Price new product [SEK]'] = 1424

df_RP.loc[df_RP['Item']=='Tommy Hilfiger kavaj', 'Source (price new product)'] =  'zalando.se'
df_RP.loc[df_RP['Item']=='Tommy Hilfiger kavaj', 'Price new product [SEK]'] = 458

df_RP.loc[df_RP['Item']=='Signerad matchtröja', 'Source (price new product)'] =  'nike.se'
df_RP.loc[df_RP['Item']=='Signerad matchtröja', 'Price new product [SEK]'] = 1599

df_RP.loc[df_RP['Item']=='Acer Chromebook CB5-312T', 'Source (price new product)'] =  'netonnet.se'
df_RP.loc[df_RP['Item']=='Acer Chromebook CB5-312T', 'Price new product [SEK]'] = 6490

df_RP.loc[df_RP['Item']=='Fönster nya 4 st. 100 cm x 1625 cm', 'Source (price new product)'] =  'hemfint.se'
df_RP.loc[df_RP['Item']=='Fönster nya 4 st. 100 cm x 1625 cm', 'Price new product [SEK]'] = 4395

df_RP.loc[df_RP['Item']=='Kemppi minarc 220', 'Source (price new product)'] =  'svetsochtillbehor.se'
df_RP.loc[df_RP['Item']=='Kemppi minarc 220', 'Price new product [SEK]'] = 16900

df_RP.loc[df_RP['Item']=='Ferguson 65 med lastare och snökedjor', 'Source (price new product)'] =  'mascus.se'
df_RP.loc[df_RP['Item']=='Ferguson 65 med lastare och snökedjor', 'Price new product [SEK]'] = 39818

df_RP.loc[df_RP['Item']=='Soffbord glas och mässing', 'Source (price new product)'] =  'royaldesign.se'
df_RP.loc[df_RP['Item']=='Soffbord glas och mässing', 'Price new product [SEK]'] = 5610

df_RP.loc[df_RP['Item']=='Märklin', 'Source (price new product)'] =  'cdon.se'
df_RP.loc[df_RP['Item']=='Märklin', 'Price new product [SEK]'] = 3899

df_RP.loc[df_RP['Item']=='Uthyres du som har stort häst intresse', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Uthyres du som har stort häst intresse', 'Price new product [SEK]'] = 1.5*6300

df_RP.loc[df_RP['Item']=='Kondens Torktumlare Av AEG LAVATHERM', 'Source (price new product)'] =  'computersalg.se'
df_RP.loc[df_RP['Item']=='Kondens Torktumlare Av AEG LAVATHERM', 'Price new product [SEK]'] = 8234

df_RP.loc[df_RP['Item']=='Älgstudsare', 'Source (price new product)'] =  'torsbohandels.se'
df_RP.loc[df_RP['Item']=='Älgstudsare', 'Price new product [SEK]'] = 7999

df_RP.loc[df_RP['Item']=='Yamaha Aerox Naked R 2014', 'Source (price new product)'] =  'ahlqvistmc.se'
df_RP.loc[df_RP['Item']=='Yamaha Aerox Naked R 2014', 'Price new product [SEK]'] = 27490

df_RP.loc[df_RP['Item']=='Helt nya Rukka Gore-Tex Byxor', 'Source (price new product)'] =  'tradeinn.se'
df_RP.loc[df_RP['Item']=='Helt nya Rukka Gore-Tex Byxor', 'Price new product [SEK]'] = 8902

df_RP.loc[df_RP['Item']=='SMC Wildcat 500 -15', 'Source (price new product)'] =  'pvlv.se'
df_RP.loc[df_RP['Item']=='SMC Wildcat 500 -15', 'Price new product [SEK]'] = 64900

df_RP.loc[df_RP['Item']=='Marshall DSL40C Vintage', 'Source (price new product)'] =  'soundstorexl.se'
df_RP.loc[df_RP['Item']=='Marshall DSL40C Vintage', 'Price new product [SEK]'] = 7899

df_RP.loc[df_RP['Item']=='Villa uthyres på Kärna, 3750 sek/mån', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Villa uthyres på Kärna, 3750 sek/mån', 'Price new product [SEK]'] = 1.5*3750

df_RP.loc[df_RP['Item']=='Apple Watch 3 42mm Cellular Milanese', 'Source (price new product)'] =  'computersalg.se'
df_RP.loc[df_RP['Item']=='Apple Watch 3 42mm Cellular Milanese', 'Price new product [SEK]'] = 6765

df_RP.loc[df_RP['Item']=='SLP pipa och slutburk', 'Source (price new product)'] =  'skoterdelen.se'
df_RP.loc[df_RP['Item']=='SLP pipa och slutburk', 'Price new product [SEK]'] = 8995

df_RP.loc[df_RP['Item']=='Lynx Adventure LX 600 ACE-11', 'Source (price new product)'] =  'brplynx.com'
df_RP.loc[df_RP['Item']=='Lynx Adventure LX 600 ACE-11', 'Price new product [SEK]'] = 134900

df_RP.loc[df_RP['Item']=='Edea Overture strl 210, vanligt storlek 31,5', 'Source (price new product)'] =  'xxl.se'
df_RP.loc[df_RP['Item']=='Edea Overture strl 210, vanligt storlek 31,5', 'Price new product [SEK]'] = 4689

df_RP.loc[df_RP['Item']=='Kit för Ac rep och fyllning', 'Source (price new product)'] =  'estimated'
df_RP.loc[df_RP['Item']=='Kit för Ac rep och fyllning', 'Price new product [SEK]'] = 1.5*2900

df_RP.loc[df_RP['Item']=='El truck -01', 'Source (price new product)'] =  'maskinexperten.se'
df_RP.loc[df_RP['Item']=='El truck -01', 'Price new product [SEK]'] = 294900

In [ ]:
df_RP['Value retained (%)'] = (df_RP['Price used product [SEK]'] / df_RP['Price new product [SEK]'])*100

In [ ]:
#reorder columns

df_RP = df_RP[['Item',	'Price used product [SEK]',	'Price new product [SEK]', 'Value retained (%)', 'Source (price new product)', 'Category',	'Region']]
df_RP.style.hide_index()

In [ ]:
#save df_RP to csv
df_RP.to_csv('processed/representative_products.csv', index=False)

In [ ]:
#save df_RP to excell to include in the supplementary information IV
df_RP.to_excel('processed/representative_products_to_SI_IV.xlsx', index=False)

In [ ]:
df_en_described = pd.read_csv(r'processed/df_cleaned_en_described.csv')
df_en_described

In [ ]:
df_en_RP_described_merged = pd.merge(df_RP, df_en_described, how='inner', on = 'Category')
df_en_RP_described_merged

In [ ]:
#keeping only the columns of interest
df_en_RP_method = df_en_RP_described_merged[['Item', 
    'Price used product [SEK]',	
    'Price new product [SEK]',	
    'Value retained (%)',	
    'Source (price new product)',	
    'Category']] 

#renaming the column
df_en_RP_method = df_en_RP_method.rename({
    'Item': 'Representative product',
    'Price used product [SEK]': 'Resale price [SEK]',
    'Price new product [SEK]': 'Retail price [SEK]', 
    'Source (price new product)': 'Source (retail price)'}, 
    axis=1)

#reorder columns
df_en_RP_method = df_en_RP_method[['Category',
    'Representative product',
    'Resale price [SEK]',
    'Retail price [SEK]',
    'Value retained (%)',
    'Source (retail price)']]
df_en_RP_method

In [ ]:
#save for including in the method section of the article
df_en_RP_method.to_excel('processed/df_en_RP_method.xlsx')

In [ ]:
#check point
# load dataframes if needed
df_en = pd.read_csv('processed/df_cleaned_en.parquet') #scraped data
df_RP = pd.read_csv('processed/representative_products.csv') #representative products

In [ ]:
#renaming column price
df_en = df_en.rename(columns = {'Price':'Resale price [SEK]'}) 

In [ ]:
# drop columns not needed
df_RP = df_RP.drop(['Item',	'Price used product [SEK]',	'Price new product [SEK]', 'Source (price new product)', 'Region'], axis=1)

In [ ]:
df_UP = df_en.merge(df_RP, on='Category', how='left')
df_UP


In [ ]:
#The retail prices are computed by dividing the prices of the secondhand items by the value retained (%) of the reference products of each category.
#df['Values'] = df.Price * df.Value_retained
df_UP['Retail price [SEK] (projected)'] = (df_UP['Resale price [SEK]'] / df_UP['Value retained (%)'])*100
df_UP

In [ ]:
#compute economic savings of buyers
df_UP['Savings of buyers [SEK]'] = df_UP['Retail price [SEK] (projected)'] - df_UP['Resale price [SEK]']

#drop column Average retail price [SEK] (projected)
df_UP.drop(columns=['Retail price [SEK] (projected)'], inplace=True)

#rename as necessary
df_UP = df_UP.rename(columns={'Resale price [SEK]': 'Earnings of sellers [SEK]'})

df_UP

In [ ]:
#save df_UP to csv
df_UP.to_csv('processed/resale_retail_product_prices.csv', index=False) #this df will be used in the calculation of the potencial environmental benetis

In [ ]:
#load df_UP if necessary
df_UP = pd.read_csv('processed/resale_retail_product_prices.csv')
df_UP

In [ ]:

# Group by 'Category' and ompute statistics
df_grouped = df_UP.groupby('Category').agg({
    'Earnings of sellers [SEK]': ['count', 'min', lambda x: x.quantile(0.25),'median', lambda x: x.quantile(0.75),'max','sum'],
    'Savings of buyers [SEK]': ['min', lambda x: x.quantile(0.25),'median', lambda x: x.quantile(0.75),'max','sum']
}).reset_index()

# Rename the columns for clarity
df_grouped.columns = [
    'Category',
    '# of products',
    'Min_e',
    '25%_e',
    '50%_e',
    '75%_e',
    'Max_e',
    'Sum_e_MSEK',
    'Min_s',
    '25%_s',
    '50%_s',
    '75%_s',
    'Max_s',
    'Sum_s_MSEK'
]

# Calculate additional columns
df_grouped['% of the total # of products'] = (df_grouped['# of products'] / df_grouped['# of products'].sum()) * 100
df_grouped['Rank by # of products'] = df_grouped['# of products'].rank(ascending=False)
df_grouped['Rank by earnings'] = df_grouped['Sum_e_MSEK'].rank(ascending=False)
df_grouped['Rank by savings'] = df_grouped['Sum_s_MSEK'].rank(ascending=False)
df_grouped['% of total earnings'] = (df_grouped['Sum_e_MSEK'] / df_grouped['Sum_e_MSEK'].sum()) * 100
df_grouped['% of total savings'] = (df_grouped['Sum_s_MSEK'] / df_grouped['Sum_s_MSEK'].sum()) * 100

#convert SEK to M SEK
df_grouped['Sum_e_MSEK'] = df_grouped['Sum_e_MSEK'] / 1e6
df_grouped['Sum_s_MSEK'] = df_grouped['Sum_s_MSEK'] / 1e6

## Convert the rank values to integers
df_grouped['Rank by # of products'] = df_grouped['Rank by # of products'].astype(int)
df_grouped['Rank by earnings'] = df_grouped['Rank by earnings'].astype(int)
df_grouped['Rank by savings'] = df_grouped['Rank by savings'].astype(int)


# Rearrange columns
df_grouped = df_grouped[[
    'Category',
    '# of products',
    'Rank by # of products',
    '% of the total # of products',
    'Min_e',
    '25%_e',
    '50%_e',
    '75%_e',
    'Max_e',
    'Sum_e_MSEK',
    'Rank by earnings',
    '% of total earnings',
    'Min_s',
    '25%_s',
    '50%_s',
    '75%_s',
    'Max_s',
    'Sum_s_MSEK',
    'Rank by savings',
    '% of total savings'
]]

#rounding
df_grouped = df_grouped.round(2)

# Sort by 'Sum_Earnings_SEK' in descending order
df_grouped = df_grouped.sort_values(by='Sum_e_MSEK', ascending=False)

# Display the resulting DataFrame
df_grouped


In [ ]:
#calculate skewness and kurtosis
from scipy.stats import skew, kurtosis

# Function to calculate skewness
def calculate_skew(series):
    return skew(series)

# Function to calculate kurtosis
def calculate_kurtosis(series):
    return kurtosis(series)

# Calculate skewness and kurtosis for 'Earnings of sellers [SEK]' and 'Savings of buyers [SEK]'
skew_earnings = df_UP.groupby('Category')['Earnings of sellers [SEK]'].agg(calculate_skew)
kurtosis_earnings = df_UP.groupby('Category')['Earnings of sellers [SEK]'].agg(calculate_kurtosis)

skew_savings = df_UP.groupby('Category')['Savings of buyers [SEK]'].agg(calculate_skew)
kurtosis_savings = df_UP.groupby('Category')['Savings of buyers [SEK]'].agg(calculate_kurtosis)

# Merge skewness and kurtosis into the existing DataFrame
df_grouped = pd.merge(df_grouped, skew_earnings.rename('Skew_Earnings'), on='Category')
df_grouped = pd.merge(df_grouped, kurtosis_earnings.rename('Kurtosis_Earnings'), on='Category')

df_grouped = pd.merge(df_grouped, skew_savings.rename('Skew_Savings'), on='Category')
df_grouped = pd.merge(df_grouped, kurtosis_savings.rename('Kurtosis_Savings'), on='Category')

# Display the resulting DataFrame
df_grouped


In [ ]:
#save df
df_grouped.reset_index(drop=True, inplace=True)
df_grouped.to_excel('processed/econ_ben_analysis.xlsx', index=False)

#table for the results/ section

In [ ]:
df_up_cat_grouped = df_UP.groupby(['Category'])
df_up_cat_mean = df_up_cat_grouped[['Resale price [SEK]', 'Retail price [SEK] (projected)']].agg('mean')
df_up_cat_mean.reset_index(inplace=True)

#rename
df_up_cat_mean = df_up_cat_mean.rename(columns = {'Resale price [SEK]':'Average resale price [SEK]'}) 
df_up_cat_mean = df_up_cat_mean.rename(columns = {'Retail price [SEK] (projected)':'Average retail price [SEK] (projected)'}) 

df_up_cat_mean

In [ ]:
#save df
df_up_cat_mean.reset_index(drop=True, inplace=True)
df_up_cat_mean.to_csv('processed/average_prices_per_category.csv', index=False)

In [ ]:
#load df if nedeed
df_up_cat_mean = pd.read_csv('processed/average_prices_per_category.csv')

In [ ]:
# load pco dataset
pco = pd.read_excel('processed/product_categories_overview.xlsx')

#keep columns of interest

columns_to_keep = ['Category',
    '# of products',
    'Rank by # of products',	
    '% of the total # of products']
pco = pco[columns_to_keep]

pco

In [ ]:
df_PC_results/_average = pco.merge(df_up_cat_mean, on='Category', how='left')
df_PC_results/_average

In [ ]:
#compute average economic savings of buyers
df_PC_results/_average['Average savings of buyers [SEK]'] = df_PC_results/_average['Average retail price [SEK] (projected)'] - df_PC_results/_average['Average resale price [SEK]']

#drop column Average retail price [SEK] (projected)
df_PC_results/_average.drop(columns=['Average retail price [SEK] (projected)'], inplace=True)

#rename as necessary
df_PC_results/_average = df_PC_results/_average.rename(columns={'Average resale price [SEK]': 'Average earnings of sellers [SEK]'})

df_PC_results/_average

In [ ]:
#adding ranks
df_PC_results/_average['Rank by average earnings of sellers [SEK]'] = df_PC_results/_average['Average earnings of sellers [SEK]'].rank(ascending=False)
df_PC_results/_average['Rank by average savings of buyers [SEK]'] = df_PC_results/_average['Average savings of buyers [SEK]'].rank(ascending=False)

## Convert the rank values to integers
df_PC_results/_average['Rank by average earnings of sellers [SEK]'] = df_PC_results/_average['Rank by average earnings of sellers [SEK]'].astype(int)
df_PC_results/_average['Rank by average savings of buyers [SEK]'] = df_PC_results/_average['Rank by average savings of buyers [SEK]'].astype(int)

#reorder columns
desired_order = ['Category',
    '# of products',
    'Rank by # of products',	
    '% of the total # of products',
    'Average earnings of sellers [SEK]',
    'Rank by average earnings of sellers [SEK]', 
    'Average savings of buyers [SEK]',
    'Rank by average savings of buyers [SEK]']
df_PC_results/_average = df_PC_results/_average[desired_order]

df_PC_results/_average

In [ ]:
#save to csv
df_PC_results/_average.to_csv('processed/PC_ecom_ben_average.csv', index=False)

## 04 Household CF Factors (SE multipliers)

In [ ]:
# TODO: implement build_or_load_household_factors(...)
# hh_cf = build_or_load_household_factors(RAW, FAC)
# write_df(hh_cf, FAC/'Y_HH_COICOP_perc_CF_SE.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
import pymrio
import pandas as pd
import numpy as np
import sys as os

In [ ]:
#loading EXIOBASE3
exio3 = pymrio.parse_exiobase3('/Users/rafa/Downloads/IOT_2022_pxp.zip') 

In [ ]:
#calculate missing matrixes
exio3.calc_all()

In [ ]:
# constants
M_EUR=10**6
EUR_TO_SEK = 10.815 # average value dor 2022 https://www.statista.com/statistics/1118911/euro-to-sek-average-monthly-exchange-rate/

In [ ]:
#characterization matrix adapted from Ivanova and Wood (2020)
characterization = pd.read_csv(r'characterization_factors/EXIOBASE341f_CC500f.csv',',', header=[0], index_col = [0])

In [ ]:
#loading corresponsence file to match Exio and COICOP01
#matrix made based on Steen-Olsen, K., Wood, R., Hertwich, E.G., 2016. The Carbon Footprint of Norwegian Household Consumption 1999–2012. J. Ind. Ecol. 20, 582–592. https://doi.org/10.1111/jiec.12405

conc = pd.read_excel(r'Concordance/Exio_COICOP12.xlsx', header=[0], index_col = [0])

In [ ]:
#final consumption expenditure in Sweden by all groups (columns)
#M€
exio3.Y.SE.groupby(level=1, sort=False).sum()

In [ ]:
#Summing the final consumption by all groups
Y_all = exio3.Y.SE.groupby(level=1, sort=False).sum()
Y_all = Y_all.sum(axis=1)
Y_all = pd.DataFrame(Y_all)
Y_all.columns = ['Final consumption expenditure by all groups']
Y_all

In [ ]:
# final consumption by all groups in COICOP format
Y_all_COICOP = Y_all.T.dot(conc.T).T
Y_all_COICOP

In [ ]:
# Final consumption expenditure by households
# values shown in M.EUR
Y_HH = exio3.Y.SE.groupby(level=1, sort=False).sum()[['Final consumption expenditure by households']]
#Y.to_csv('Y_SE_hh.csv')
Y_HH

In [ ]:
# Final consumption expenditure by households in COICOP format
# values shown in M.EUR
Y_HH_COICOP = Y_HH.T.dot(conc.T).T
Y_HH_COICOP

In [ ]:
# multiplication factor (the percentage that the final consumption by househould has from the final consumption of all groups)
HH_Y_factor_exio = Y_HH.div(Y_all['Final consumption expenditure by all groups'], axis=0)
HH_Y_factor_exio

In [ ]:
# multiplication factor (the percentage that the final consumption by househould has from the final consumption of all groups)
HH_Y_factor_COICOP = Y_HH_COICOP.div(Y_all_COICOP['Final consumption expenditure by all groups'], axis=0)
HH_Y_factor_COICOP

In [ ]:
#rounding the sum of the first column
Y_sum = round(Y_HH['Final consumption expenditure by households'].sum(), 9) #needed to round the sum of the values
Y_COICOP_sum = round(Y_HH_COICOP['Final consumption expenditure by households'].sum(), 9)

if Y_COICOP_sum == Y_sum:
    print('The aggregation of the consumption values in the matris Y_HH_COICOP is CORRECT')
else:
    print('The aggregation of the consumption values in the matris Y_HH_COICOP is INCORRECT')

In [ ]:
if Y_HH_COICOP.sum().all() == Y_HH.sum().all():
    print('The aggregation of the consumption values in the matris Y_HH_COICOP is CORRECT')
else:
    print('The aggregation of the consumption values in the matris Y_HH_COICOP is INCORRECT')

In [ ]:
if Y_all_COICOP.sum().all() == Y_all.sum().all():
    print('The aggregation of the consumption values in the matris Y_all is CORRECT')
else:
    print('The aggregation of the consumption values in the matris Y_all is INCORRECT')

In [ ]:
#matrix comsumption-based accounting
#stressors from total consumption in SE from all groups

# values shown in stressors per M.EUR

D_cba_all = exio3.satellite.D_cba.SE.T
D_cba_all 

In [ ]:
# Converting to COICOP format
# values shown in stressors per M.EUR

D_cba_all_COICOP = D_cba_all.T.dot(conc.T).T
D_cba_all_COICOP

In [ ]:
#Checking if the aggregation is correct

if D_cba_all_COICOP.sum().all() == D_cba_all.sum().all():
    print('The aggregation of the consumption values in the matris D_cba_all is CORRECT')
else:
    print('The aggregation of the consumption values in the matris Y_cba_all is INCORRECT')

In [ ]:
# aligning the matrixes
df1 = characterization.copy()
df2 = D_cba_all.copy()

characterization = df1.drop([idx for idx in df1.index if idx in df1.index and idx not in df2.columns], axis=0)
D_cba_all = df2.drop([col for col in df2.columns if col in df2.columns and col not in df1.index], axis=1)

In [ ]:
##Converting stressors into footprints
#this is also to check it the ggregation of the fotprints is correct
D_cba_all_cf = D_cba_all.dot(characterization)
D_cba_all_cf

In [ ]:
# aligning the matrixes
#df1 = characterization.copy()
df3 = D_cba_all_COICOP.copy()

#characterization = df1.drop([idx for idx in df1.index if idx in df1.index and idx not in df2.columns], axis=0)
D_cba_all_COICOP = df3.drop([col for col in df3.columns if col in df3.columns and col not in df1.index], axis=1)

In [ ]:
#Converting stressors into footprints
D_cba_all_COICOP_cf = D_cba_all_COICOP.dot(characterization)
D_cba_all_COICOP_cf

In [ ]:
#Checking if the aggregation is correct

if D_cba_all_COICOP_cf.sum().all() == D_cba_all_cf.sum().all():
    print('The aggregation of the consumption values in the matris D_cba_all_COICOP_cf is CORRECT')
else:
    print('The aggregation of the consumption values in the matris D_cba_all_COICOP_cf is INCORRECT')

In [ ]:
#multiplying each of the footprints of all groups by the the percentage that the final consumption by househould has from the final consumption of all groups

D_cba_hh_cf_exio = D_cba_all_cf[['GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']].multiply(HH_Y_factor_exio['Final consumption expenditure by households'], axis='index')
D_cba_hh_cf_exio

In [ ]:
#multiplying each of the footprints of all groups by the the percentage that the final consumption by househould has from the final consumption of all groups

D_cba_hh_cf_COICOP = D_cba_all_COICOP_cf[['GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']].multiply(HH_Y_factor_COICOP['Final consumption expenditure by households'], axis='index')
D_cba_hh_cf_COICOP

In [ ]:
# finding the environmental intensity factors of household expenditure
# Divide the the matrix impact of total consumption (D_cba_hh_cf) by the matrix final consumption expenditure by households (Y_HH)

# footprints per M.EUR

multipliers_exio = D_cba_hh_cf_exio.div(Y_HH['Final consumption expenditure by households'], axis=0)

#renaiming columns
#renaming columns in pandas https://stackoverflow.com/questions/11346283/renaming-columns-in-pandas
multipliers_exio.columns = ['Carbon Footprint multiplier [kg CO2-eq. per M€]']

#multipliers_exio.to_excel('multipliers_CH_exio.xlsx')
multipliers_exio

In [ ]:
# finding the environmental intensity factors of household expenditure
# Divide the the matrix impact of total consumption (D_cba_hh_cf) by the matrix final consumption expenditure by households (Y_HH)


# footprints per M.EUR

multipliers_COICOP = D_cba_hh_cf_COICOP.div(Y_HH_COICOP['Final consumption expenditure by households'], axis=0)

#renaiming columns
#renaming columns in pandas https://stackoverflow.com/questions/11346283/renaming-columns-in-pandas
multipliers_COICOP.columns = ['Carbon Footprint multiplier [kg CO2-eq. per M€]']

#multipliers_exio.to_excel('multipliers_SE_exio.xlsx')
multipliers_COICOP

In [ ]:
#Adjusting the currency and the unit

multipliers_COICOP_SEK = multipliers_COICOP / EUR_TO_SEK / 1000000
multipliers_COICOP_SEK.columns = ['Carbon Footprint multiplier [kg CO2-eq. per SEK]']
multipliers_COICOP_SEK

In [ ]:
# Merging the matrices Y_HH and multipliers_exio
Y_multi_exio = Y_HH.merge(multipliers_exio, on='sector')
Y_multi_exio.rename(columns = {'Final consumption expenditure by households': 'Final consumption expenditure by households [M€]'}, inplace=True)
Y_multi_exio

In [ ]:
#Ajusting the currency of the Y_HH_COICOP
Y_HH_COICOP_SEK = Y_HH_COICOP / EUR_TO_SEK
Y_HH_COICOP_SEK.rename(columns = {'Final consumption expenditure by households': 'Final consumption expenditure by households [M.SEK]'}, inplace=True)
Y_HH_COICOP_SEK

In [ ]:
# Merging the matrices Y_HH_COICOP and multipliers_COICOP
Y_multi_COICOP = Y_HH_COICOP_SEK.merge(multipliers_COICOP_SEK, on='Sector')
Y_multi_COICOP

In [ ]:
# define a custom function that takes the names of the columns of our data and calculates the weighted average.
#Then, use apply to execute it against our grouped data.
#https://pbpython.com/weighted-average.html

def wavg(group, avg_name, weight_name):
    d = group[avg_name]
    w = group[weight_name]
    try:
        return (d * w).sum() / w.sum()
    except ZeroDivisionError:
        return d.mean()

In [ ]:
# weighted average all products COICOP
# Kg CO2 per SEK spent

wavg(Y_multi_COICOP, 'Carbon Footprint multiplier [kg CO2-eq. per SEK]', 'Final consumption expenditure by households [M.SEK]')

In [ ]:
# In %s
#In [1371]: (100. * df / df.sum()).round(0)
#Out[1371]:

#Y_perc = (100. * Y_COICOP / Y_COICOP.sum()).round(2).astype(str) + '%'
Y_HH_COICOP_perc = (Y_HH_COICOP / Y_HH_COICOP.sum() * 100).round(4)
Y_HH_COICOP_perc.columns = ['Final consumption expenditure by households [ratio]'] #rename column
Y_HH_COICOP_perc

In [ ]:
# Merging the two matrices
Y_HH_COICOP_perc_CF = Y_HH_COICOP_perc.merge(multipliers_COICOP_SEK['Carbon Footprint multiplier [kg CO2-eq. per SEK]'], on='Sector', how='left')
#Y_HH_COICOP_perc_CF.to_csv('Y_HH_COICOP_perc_CF.csv')
Y_HH_COICOP_perc_CF
#wavg_Y.to_csv('wavg_Y.csv')


#Plot in R

In [ ]:
Y_HH_COICOP_perc_CF.to_csv('processed/Y_HH_COICOP_perc_CF_SE.parquet')

# plot chart in R. 

In [ ]:
Y = exio3.Y.SE.groupby(level=1, sort=False).sum()[['Final consumption expenditure by households']]
Y

In [ ]:
D_cba_SE = exio3.satellite.D_cba.SE.T

In [ ]:
df1 = characterization.copy()
df2 = D_cba_SE.copy()

characterization = df1.drop([idx for idx in df1.index if idx in df1.index and idx not in df2.columns], axis=0)
D_cba_SE = df2.drop([col for col in df2.columns if col in df2.columns and col not in df1.index], axis=1)

In [ ]:
# Impact of total consumption per sector
# Impact per M.EUR (total consumption)

D_cba_SE_footprints = D_cba_SE.dot(characterization)

In [ ]:
D_cba_SE_footprints[['GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']]

In [ ]:
# Dividing the matrix impact of total consumption (D_cba_SE) by the matrix total consumption (Y) to find the consumption intensity factors
# CO2 per M.EUR
co2_per_MEUR = D_cba_SE_footprints[['GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']].div(Y['Final consumption expenditure by households'], axis=0)
co2_per_MEUR

In [ ]:
# Merging the two matrices
multi = Y.merge(co2_per_MEUR, on='sector')
multi

In [ ]:
# canculating the weighted average
#(df["quantity"] * df["weight"]).sum() / df["weight"].sum()
# kg per euros
household_emissions_per_EUR = (multi['GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']*multi['Final consumption expenditure by households']).sum()/multi['Final consumption expenditure by households'].sum()/M_EUR
household_emissions_per_EUR

In [ ]:
household_emissions_per_SEK = household_emissions_per_EUR/EUR_TO_SEK*INFLATION_EUR_2011_2020

In [ ]:
print(str(household_emissions_per_SEK) + ' [Kg CO\u2082eq]/[SEK]') 

## 05 Potential Environmental Benefits (IO‑LCA)

In [ ]:
# TODO: implement load_exio_factors(...), compute_potential_env_benefits(...)
# econ_ben = pd.read_parquet(DER/'PC_ecom_ben.parquet')
# prices = pd.read_parquet(DER/'resale_retail_product_prices.parquet')
# exio_cf, mappings = load_exio_factors(FAC)
# pot_env_ben = compute_potential_env_benefits(econ_ben, prices, exio_cf, mappings)
# write_df(pot_env_ben, DER/'SI_V_pot_env_ben.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
#importing pymrio library
import pymrio

In [ ]:
#loading EXIOBASE3
exio3 = pymrio.parse_exiobase3('/Users/rafa/Downloads/IOT_2022_pxp.zip') 

In [ ]:
#calculate missing matrixes
exio3.calc_all()

In [ ]:
#chatacterisation factors to convert environmemtal stressors into footprints - adapted from Ivanova and Wood (2020) 
characterization = pd.read_csv(r'characterization_factors/EXIOBASE341f_CC500f.csv',',', header=[0], index_col = [0])

In [ ]:
# Multipliers (total, direct and indirect, requirement factors for one unit of output) for Sweden

M_SE = exio3.satellite.M.SE.T

In [ ]:
#removing stressors that are not in the characterization matrix
df1 = characterization.copy()
df2 = M_SE.copy()

characterization = df1.drop([idx for idx in df1.index if idx in df1.index and idx not in df2.columns], axis=0)
M_SE = df2.drop([col for col in df2.columns if col in df2.columns and col not in df1.index], axis=1)

In [ ]:
#converting the stressors into environmental impacts.
M_SE_footprints = M_SE.dot(characterization)

In [ ]:
#saving the dataframe
M_SE_footprints.to_excel(r'processed/footprints_SE.xlsx')

In [ ]:
# changing frist column for not being the index of the dataframe
M_SE_footprints.reset_index(inplace=True)

In [ ]:
# carbon footprint per consumption category for Sweden

#keeping only the columns of interest
CF_exio = M_SE_footprints[['sector', 'GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)']] 
#renaming the column
CF_exio = CF_exio.rename({'GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)': 'GHG emissions per M EUR'}, axis=1)
CF_exio

In [ ]:
#saving to an excel file
CF_exio.to_excel(r'processed/CF_exio_RL.xlsx')

In [ ]:
# load mach file of blocket and exio categories 
df_32cat_CFexio = pd.read_excel(r'blocket_exio_match/df_32cat_CFexio_RL.xlsx') 

# Multiply to 10e6 to convert MEUR to EUR and convert EUR to SEK 
M_EUR=10**6
EUR_TO_SEK = 11.025436 #https://www.xe.com/ 20221216
df_32cat_CFexio['Exio product [GHG emissions per M EUR]'] = df_32cat_CFexio['Exio product [GHG emissions per M EUR]'] / (EUR_TO_SEK*M_EUR)

# rename column
df_32cat_CFexio = df_32cat_CFexio.rename({'Exio product [GHG emissions per M EUR]': 'Exio product [kg CO2eq. per SEK]'}, axis=1)
df_32cat_CFexio

In [ ]:
# calculating the average of CO2 emissions multipliers of categories
## groupby columns Col1 and estimate the mean of column Col2
#df.groupby([Col1])[Col2].mean()

df_32cat_CFexio_mean_cat = df_32cat_CFexio.groupby(['Category'], as_index=False)['Exio product [kg CO2eq. per SEK]'].mean()
df_32cat_CFexio_mean_cat = df_32cat_CFexio_mean_cat.rename({'Exio product [kg CO2eq. per SEK]': 'MRIO multiplier [kg CO2eq. per SEK]'}, axis=1)
df_32cat_CFexio_mean_cat

In [ ]:
#save
#include the dataset in supporting information IV
df_32cat_CFexio_mean_cat.to_excel(r'processed/df_32cat_CFexio_mean_cat_to_SI_IV.xlsx')

In [ ]:
import pandas as pd

# load the dataframe if necessary
df_32cat_CFexio_mean_cat = pd.read_excel('processed/df_32cat_CFexio_mean_cat_to_SI_IV.xlsx')
#cleaning df
df_32cat_CFexio_mean_cat = df_32cat_CFexio_mean_cat.drop(columns=['Unnamed: 0'])

df_UP = pd.read_csv('processed/resale_retail_product_prices.csv') #from PC_ecom_ben

In [ ]:
df_UP

In [ ]:
# adding the multipliers to the UP dataframe

df_EB = df_UP.merge(df_32cat_CFexio_mean_cat, on='Category', how='left')
df_EB

In [ ]:
# calculating the environmental benefits

# impacts of used products (economic allocation approach)
df_EB['Used product CF [kg CO2eq.] EAA'] = df_EB['Earnings of sellers [SEK]'] * df_EB['MRIO multiplier [kg CO2eq. per SEK]'] 

# cradle-to-gate emissions (this is equal to avoided emissions in the zero-burden approach)
df_EB['New product CF [kg CO2eq.] EBZBA'] = df_EB['Retail price [SEK] (projected)'] * df_EB['MRIO multiplier [kg CO2eq. per SEK]'] 

# Environmental benefits using an economic allocation approach
df_EB['EBEAA [kg CO2eq.]'] = df_EB['New product CF [kg CO2eq.] EBZBA'] - df_EB['Used product CF [kg CO2eq.] EAA'] 

# reorder umns
df_EB = df_EB[['Item', 'Category',	'Region', 'Resale price [SEK]', 'Retail price [SEK] (projected)', 'Value retained (%)',
    'MRIO multiplier [kg CO2eq. per SEK]', 'Used product CF [kg CO2eq.] EAA',	'New product CF [kg CO2eq.] EBZBA',	'EBEAA [kg CO2eq.]']]

df_EB

In [ ]:
# save dataframe
df_EB.to_csv(r'processed/SI_V_pot_env_ben.parquet', index=False)

In [ ]:
#check point
#load dataset if needed
df_EB = pd.read_csv(r'processed/SI_V_pot_env_ben.parquet')

In [ ]:
df_EB

In [ ]:
df_EB_cat_describe = df_EB.groupby(['Category'])['Used product CF [kg CO2eq.] EAA', 'New product CF [kg CO2eq.] EBZBA', 'EBEAA [kg CO2eq.]'].describe()
df_EB_cat_describe

In [ ]:
#save
df_EB_cat_describe.to_excel('processed/SI_VI_df_EB_cat_describe.xlsx')

In [ ]:
# sum of EB of categories
#https://stackoverflow.com/questions/48909110/python-pandas-mean-and-sum-groupby-on-different-columns-at-the-same-time
df_EB_cat = df_EB.groupby('Category', as_index=False).agg(EBZBA_category_sum=('New product CF [kg CO2eq.] EBZBA','sum'),
    EBEAA_category_sum=('EBEAA [kg CO2eq.]','sum'),
    EBZBA_category_mean=('New product CF [kg CO2eq.] EBZBA','mean'),
    EBEAA_category_mean=('EBEAA [kg CO2eq.]','mean'))

# converte kg to tonnes EB
df_EB_cat['EBZBA_category_sum']=df_EB_cat['EBZBA_category_sum']/1000
df_EB_cat['EBEAA_category_sum']=df_EB_cat['EBEAA_category_sum']/1000

# calculate fold difference sum EB approaches
df_EB_cat['Fold difference EB approaches'] = df_EB_cat['EBZBA_category_sum'] / df_EB_cat['EBEAA_category_sum']

# renaming columns
df_EB_cat = df_EB_cat.rename(columns = {'EBZBA_category_sum':'Total EBZBA [tonnes CO2eq.]',
    'EBEAA_category_sum': 'Total EBEAA [tonnes CO2eq.]',
    'EBZBA_category_mean':'Average EBZBA [kg CO2eq.]',
    'EBEAA_category_mean': 'Average EBEAA [kg CO2eq.]'})

# sort by Total EBZBA
df_EB_cat = df_EB_cat.sort_values(by =['Total EBZBA [tonnes CO2eq.]'], ascending=False)


df_EB_cat.style.hide_index() 

In [ ]:
# save
df_EB_cat.to_excel(r'processed/pot_env_ben.xlsx', index=False)

In [ ]:
# load ecom_ben dataset
PC_ecom_ben = pd.read_csv('processed/PC_ecom_ben.parquet') #created in SIxx_PC_ecom_ben.ipynb

PC_ecom_ben

In [ ]:
# merge datasets
df_results/ = PC_ecom_ben.merge(df_EB_cat, on='Category', how='left')
df_results/

In [ ]:
#drop columns not of interest
df_results/_copy = df_results/.copy()
df_results/_copy = df_results/_copy.drop(columns=['Average EBZBA [kg CO2eq.]',	'Average EBEAA [kg CO2eq.]'])
df_results/_copy

In [ ]:
#include rank for EB
df_results/_copy['Rank by Total EBZBA [tonnes CO2eq.]'] = df_results/_copy['Total EBZBA [tonnes CO2eq.]'].rank(ascending=False)
df_results/_copy['Rank by Total EBEAA [tonnes CO2eq.]'] = df_results/_copy['Total EBEAA [tonnes CO2eq.]'].rank(ascending=False)

# Convert the rank values to integers
df_results/_copy['Rank by Total EBZBA [tonnes CO2eq.]'] = df_results/_copy['Rank by Total EBZBA [tonnes CO2eq.]'].astype(int)
df_results/_copy['Rank by Total EBEAA [tonnes CO2eq.]'] = df_results/_copy['Rank by Total EBEAA [tonnes CO2eq.]'].astype(int)

df_results/_copy

In [ ]:
#reorder columns

# Identify the columns to move and their target positions
columns_to_move = ['Rank by Total EBZBA [tonnes CO2eq.]',	'Rank by Total EBEAA [tonnes CO2eq.]']
target_positions = [10, 12] 

# Create a list of column names in the desired order
all_columns = df_results/_copy.columns.tolist()

# Remove the columns to move from their original positions
for col in columns_to_move:
    all_columns.remove(col)

# Insert the columns to move at the target positions
for col, pos in zip(columns_to_move, target_positions):
    all_columns.insert(pos, col)

# Reorder the DataFrame based on the new column order
df_results/_copy = df_results/_copy[all_columns]

df_results/_copy


In [ ]:
#rename columns
df_results/_copy = df_results/_copy.rename({
    'Total value [M SEK] (resale)': 'Earnings of sellers [M SEK]',
    'Rank by total value (resale)': 'Rank by earnings of sellers',
    '% of total value (resale)': '% of total earnings of sellers', 
    'Total economic savings of buyers [M SEK]': 'Savings of buyers [M SEK]',
    'Rank by total economic savings': 'Rank by savings of buyers',
    'Total EBZBA [tonnes CO2eq.]': 'EBZBA [tonnes CO2eq.]',
    'Rank by Total EBZBA [tonnes CO2eq.]': 'Rank by EBZBA',
    'Total EBEAA [tonnes CO2eq.]': 'EBEAA [tonnes CO2eq.]',
    'Rank by Total EBEAA [tonnes CO2eq.]': 'Rank by EBEAA'
    }, 
    axis=1)

df_results/_copy

In [ ]:
#save and include as a table in the results/
df_results/_copy.to_excel(r'processedresults/_ecom_ecol_ben.xlsx', index=False)

## 06 Re‑spending Effects

In [ ]:
# TODO: implement compute_re_spending(...)
# econ_ben = pd.read_parquet(DER/'PC_ecom_ben.parquet')
# hh_cf = pd.read_parquet(FAC/'Y_HH_COICOP_perc_CF_SE.parquet')
# re_agg, re_disagg = compute_re_spending(econ_ben, hh_cf)
# write_df(re_agg, DER/'df_re_spending_effect_agg.parquet')
# write_df(re_disagg, DER/'df_re_spending_effect_disagg.parquet')

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
import pandas as pd
import numpy as np
import sys as os

In [ ]:
# loading data from SI_V_carbon_footprint_SE.ipynb
df_CF_multipliers = pd.read_csv(r'processed/Y_HH_COICOP_perc_CF_SE.parquet')
df_CF_multipliers

In [ ]:
# min and max CF_multipliers
CF_multiplier_min = df_CF_multipliers['Carbon Footprint multiplier [kg CO2-eq. per SEK]'].min()
CF_multiplier_min_sector = df_CF_multipliers['Sector'].loc[df_CF_multipliers['Carbon Footprint multiplier [kg CO2-eq. per SEK]'].idxmin()]
CF_multiplier_max = df_CF_multipliers['Carbon Footprint multiplier [kg CO2-eq. per SEK]'].max()
CF_multiplier_max_sector = df_CF_multipliers['Sector'].loc[df_CF_multipliers['Carbon Footprint multiplier [kg CO2-eq. per SEK]'].idxmax()]
print('CF_multiplier_min = ', CF_multiplier_min, CF_multiplier_min_sector, '\nCF_multiplier_max =', CF_multiplier_max, CF_multiplier_max_sector)

In [ ]:
# weighted average CF_multiplier
# define a custom function that takes the names of the columns of our data and calculates the weighted average.
#Then, use apply to execute it against our grouped data.
#https://pbpython.com/weighted-average.html

def wavg(group, avg_name, weight_name):
    d = group[avg_name]
    w = group[weight_name]
    try:
        return (d * w).sum() / w.sum()
    except ZeroDivisionError:
        return d.mean()

In [ ]:
CF_multiplier_wavg = wavg(df_CF_multipliers, 'Carbon Footprint multiplier [kg CO2-eq. per SEK]', 'Final consumption expenditure by households [ratio]')
CF_multiplier_wavg

In [ ]:
#economic and environmental benefits from SI_VIII_pot_env_ben
df_econ_env_ben = pd.read_excel('processedresults/_ecom_ecol_ben.xlsx')

#keeping only necessary columns
df_re_spending_effect = df_econ_env_ben[['Category', 
    'Earnings of sellers [M SEK]', 
    'Savings of buyers [M SEK]', 
    'EBZBA [tonnes CO2eq.]',
    'EBEAA [tonnes CO2eq.]']].copy()

df_re_spending_effect

In [ ]:
# calculate the sum of each column and create a resulting DataFrame with the sums

sums_df = df_re_spending_effect.drop(columns='Category').sum(axis=0)
sums_df = sums_df.to_frame().T
sums_df


In [ ]:
sums_df['Min_impact_Savings_buyers_tonnes_Kg_CO2'] = (sums_df['Savings of buyers [M SEK]'] * 1000) * CF_multiplier_min
sums_df['Max_impact_Savings_buyers_tonnes_Kg_CO2'] = (sums_df['Savings of buyers [M SEK]'] * 1000) * CF_multiplier_max
sums_df['Wavg_impact_Savings_buyers_tonnes_Kg_CO2'] = (sums_df['Savings of buyers [M SEK]'] * 1000) * CF_multiplier_wavg
sums_df

In [ ]:
sums_df['Min_impact_Earnings_sellers_tonnes_Kg_CO2'] = (sums_df['Earnings of sellers [M SEK]'] * 1000) * CF_multiplier_min
sums_df['Max_impact_Earnings_sellers_tonnes_Kg_CO2'] = (sums_df['Earnings of sellers [M SEK]'] * 1000) * CF_multiplier_max
sums_df['Wavg_impact_Earnings_sellers_tonnes_Kg_CO2'] = (sums_df['Earnings of sellers [M SEK]'] * 1000) * CF_multiplier_wavg
sums_df

In [ ]:
sums_df.to_csv(r'processed/df_re_spending_effect_agg.parquet', index=False)
#print charts in RStudio

In [ ]:
#Disaggregated results/ for Earnings of sellers and Savings of buyers from SI_VII_PC_econ_ben
df_econ_ben = pd.read_csv('processed/diagg_ecom_ben.csv')

df_econ_ben

In [ ]:
# Calculate the sum of multiple columns
sum_of_columns = df_econ_ben[['Earnings of sellers [SEK]', 'Savings of buyers [SEK]']].sum()

# Print the sum of each column
print(sum_of_columns)

In [ ]:
df_econ_ben['Min_impact_Savings_buyers_Kg_CO2'] = df_econ_ben['Savings of buyers [SEK]'] * CF_multiplier_min
df_econ_ben['Max_impact_Savings_buyers_Kg_CO2'] = df_econ_ben['Savings of buyers [SEK]'] * CF_multiplier_max
df_econ_ben['Wavg_impact_Savings_buyers_Kg_CO2'] = df_econ_ben['Savings of buyers [SEK]'] * CF_multiplier_wavg

df_econ_ben['Min_impact_Earnings_of_sellers_Kg_CO2'] = df_econ_ben['Earnings of sellers [SEK]'] * CF_multiplier_min
df_econ_ben['Max_impact_Earnings_of_sellers_Kg_CO2'] = df_econ_ben['Earnings of sellers [SEK]'] * CF_multiplier_max
df_econ_ben['Wavg_impact_Earnings_of_sellers_Kg_CO2'] = df_econ_ben['Earnings of sellers [SEK]'] * CF_multiplier_wavg

df_econ_ben

In [ ]:
df_econ_ben.to_csv(r'processed/df_re_spending_effect_disagg.parquet', index=False)

In [ ]:
df_econ_ben_des = df_econ_ben.describe()

In [ ]:
df_econ_ben_des

## 07 ERE computation

In [ ]:
# TODO: implement compute_ere(...)
# pot = pd.read_parquet(DER/'SI_V_pot_env_ben.parquet')
# re_disagg = pd.read_parquet(DER/'df_re_spending_effect_disagg.parquet')
# df_ere = compute_ere(pot, re_disagg)
# write_df(df_ere, RES/'df_ere.parquet')
# df_ere.to_csv(EXP/'df_ere.csv', index=False)  # optional export

> Imported code from previous notebooks (lightly path‑remapped)

In [ ]:
import pandas as pd

In [ ]:
# loading data from SI_VI_pot_env_ben.ipynb
results/_pco_eb = pd.read_excel(r'processedresults/_ecom_ecol_ben.xlsx')
results/_pco_eb = results/_pco_eb.sort_values(by='Rank by # of products', ascending=True)
results/_pco_eb

In [ ]:
# keep only the necessary columns
df_peb = results/_pco_eb[['Category', 'EBZBA [tonnes CO2eq.]', 'EBEAA [tonnes CO2eq.]']].copy()
df_peb

In [ ]:
# loading data from SI_VII_Re-spending_effects.ipynb
df_re_spending_effect = pd.read_csv(r'processed/df_re_spending_effect_disagg.parquet')
df_re_spending_effect

In [ ]:
# group impacts into product categories
df_pre = df_re_spending_effect.groupby(['Category'])['Min_impact_Savings_buyers_Kg_CO2', 'Max_impact_Savings_buyers_Kg_CO2', 'Wavg_impact_Savings_buyers_Kg_CO2', 
    'Min_impact_Earnings_of_sellers_Kg_CO2',	'Max_impact_Earnings_of_sellers_Kg_CO2',	'Wavg_impact_Earnings_of_sellers_Kg_CO2'].sum()
df_pre

In [ ]:
# Computing min, wavg, and max PRE; converting into tonnes

df_pre['PRE_min_tonnes_CO2'] = df_pre['Min_impact_Savings_buyers_Kg_CO2'] + df_pre['Min_impact_Earnings_of_sellers_Kg_CO2']
df_pre['PRE_wavg_tonnes_CO2'] = df_pre['Wavg_impact_Savings_buyers_Kg_CO2'] + df_pre['Wavg_impact_Earnings_of_sellers_Kg_CO2']
df_pre['PRE_max_tonnes_CO2'] = df_pre['Max_impact_Savings_buyers_Kg_CO2'] + df_pre['Max_impact_Earnings_of_sellers_Kg_CO2']

df_pre['PRE_min_tonnes_CO2'] = df_pre['PRE_min_tonnes_CO2'] / 1000
df_pre['PRE_wavg_tonnes_CO2'] = df_pre['PRE_wavg_tonnes_CO2'] / 1000
df_pre['PRE_max_tonnes_CO2'] = df_pre['PRE_max_tonnes_CO2'] / 1000
df_pre = df_pre[['PRE_min_tonnes_CO2', 'PRE_wavg_tonnes_CO2', 'PRE_max_tonnes_CO2']].copy()
df_pre

In [ ]:
# merge dfs
df_ere = df_peb.merge(df_pre, on='Category', how='left')
df_ere


In [ ]:

df_ere['ERE_%_min_EBZBA'] = df_ere['PRE_min_tonnes_CO2'] / df_ere['EBZBA [tonnes CO2eq.]'] * 100
df_ere['ERE_%_wavg_EBZBA'] = df_ere['PRE_wavg_tonnes_CO2'] / df_ere['EBZBA [tonnes CO2eq.]'] * 100
df_ere['ERE_%_max_EBZBA'] = df_ere['PRE_max_tonnes_CO2'] / df_ere['EBZBA [tonnes CO2eq.]'] * 100


df_ere['ERE_%_min_EBEAA'] = df_ere['PRE_min_tonnes_CO2'] / df_ere['EBEAA [tonnes CO2eq.]'] * 100
df_ere['ERE_%_wavg_EBEAA'] = df_ere['PRE_wavg_tonnes_CO2'] / df_ere['EBEAA [tonnes CO2eq.]'] * 100
df_ere['ERE_%_max_EBEAA'] = df_ere['PRE_max_tonnes_CO2'] / df_ere['EBEAA [tonnes CO2eq.]'] * 100

df_ere

In [ ]:
# keep collumns of interest
df_ere_save = df_ere[['Category', 'ERE_%_min_EBZBA', 'ERE_%_wavg_EBZBA', 'ERE_%_max_EBZBA', 'ERE_%_min_EBEAA', 'ERE_%_wavg_EBEAA', 'ERE_%_max_EBEAA']]
df_ere_save

In [ ]:
#save and include as a table in the results/
df_ere_save.to_excel(r'processed/df_ere.parquet', index=False)


---
### Tips
- Move logic from the **Imported code** cells into the stubs to create clean, reusable functions.
- Keep canonical artifacts immutable (`processed/df_cleaned_en.parquet`); use `enforce=True` only when you intend to overwrite.
- Prefer Parquet for speed and schema stability; reserve CSV/Excel for sharing from `exports/`.
